# Bridging the Gap Between Education and Industry
## A Machine Learning-Based Study of Career Change

**Author:** Cilveri Prashanth — B.Sc. Mathematics, Statistics and Computer Science (MSCS)
Bhavan's Vivekananda College of Science, Humanities & Commerce · Osmania University · 2025–2026

This notebook reconstructs the analysis from Appendix A of the academic report: six classifiers trained to predict **Likely to Change Occupation** from 22 education, skills and employment features.

> **Dataset required:** place the original `dataset5k.csv` in `../data/` before running. The dataset is not distributed with this repository — see `../data/README.md`. **No outputs are pre-filled in this notebook, and no result is claimed until the cells are actually run.**
>
> **One deliberate change from the report's code:** the train–test split now happens *before* scaling, and the scaler is fitted on the training data only (the report fitted it on all rows first — a data-leakage bug; see `../PROJECT_REVIEW.md` §3.9).

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc

## 2. Load the dataset

The report's code loads `dataset5k.csv` and drops incomplete rows.

In [ ]:
data = pd.read_csv('../data/dataset5k.csv')
data = data.dropna()
print(data.shape)
data.head()

## 3. Preprocessing

Label-encode the categorical columns, then separate features from the target **Likely to Change Occupation** (the target used by the report's code and confirmed in Appendix B — not the employed/unemployed outcome described in some chapters).

In [ ]:
for column in data.select_dtypes(include=['object']).columns:
    data[column] = LabelEncoder().fit_transform(data[column])

X = data.drop('Likely to Change Occupation', axis=1)
y = data['Likely to Change Occupation'].astype(int)
feature_names = list(X.columns)
print(len(feature_names), 'features')

## 4. Train–test split, then scale (leakage fixed)

Split first (80/20, `random_state=42`, as in the report), then fit `StandardScaler` on the training rows only.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print('train:', X_train.shape, ' test:', X_test.shape)

## 5. Train the six classifiers

Same models and settings as Appendix A of the report.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'SVM': SVC(probability=True),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
}

accuracy_results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accuracy_results[name] = acc
    print('\n====================================')
    print(f'Model: {name}')
    print('Accuracy:', acc)
    print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
    print('Classification Report:\n', classification_report(y_test, y_pred))

## 6. Accuracy comparison

In [ ]:
plt.figure()
plt.bar(accuracy_results.keys(), accuracy_results.values())
plt.title('Model Accuracy Comparison')
plt.xticks(rotation=45)
plt.ylabel('Accuracy')
plt.tight_layout()
plt.savefig('../reports/figures/generated_accuracy_comparison.png')
plt.show()

## 7. Confusion matrix — Random Forest

In [ ]:
rf_model = models['Random Forest']
cm = confusion_matrix(y_test, rf_model.predict(X_test))
plt.figure()
sns.heatmap(cm, annot=True, fmt='d')
plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('../reports/figures/generated_confusion_matrix_rf.png')
plt.show()

## 8. ROC curve — Random Forest

In [ ]:
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob_rf)
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, label='ROC Curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Random Forest')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../reports/figures/generated_roc_curve_rf.png')
plt.show()

## 9. Feature importance — Random Forest

Uses the **true** feature names. (The report's printed code dropped a non-existent `Employment_Status` column here — see `../PROJECT_REVIEW.md` §3.8.)

In [ ]:
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_model.feature_importances_,
}).sort_values(by='Importance', ascending=False)
print(importance_df)
plt.figure()
plt.barh(importance_df['Feature'], importance_df['Importance'])
plt.title('Feature Importance - Random Forest')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.tight_layout()
plt.savefig('../reports/figures/generated_feature_importance_rf.png')
plt.show()

## 10. Documented results from the academic report (not reproduced)

The original dataset is unavailable, so the values below — read from the report's own output screenshots (Figs. 4.1–4.6) — are the project's documented results. A different set printed in Chapter 6, Table 6.1 could not be verified against any output; both sets are compared in `../reports/model_comparison.md`.

| Model | Documented accuracy (Figs. 4.1–4.6) | Chapter 6, Table 6.1 |
|---|---|---|
| Logistic Regression | 90.28% | 78% |
| Decision Tree | 100% | 82% |
| Random Forest | 100% | 89% |
| SVM | 97.37% | 87% |
| KNN | 85.34% | 80% |
| Naive Bayes | 61.88% | 75% |

Top documented features (Fig. 6.4.1): **Job Satisfaction 0.546 · Career Change Interest 0.237 · Salary 0.146** — together ~93% of total importance, with Decision Tree and Random Forest at 100% on 5,763 test rows. That pattern indicates a synthetically generated dataset whose target is largely derived from a few inputs; the accuracies show the models recover that rule and should not be read as real-world predictive performance. See `../PROJECT_REVIEW.md` §3.10.